In [1]:
pip install gradio


Note: you may need to restart the kernel to use updated packages.


In [2]:
#Creación de App con Gradio
import gradio as gr
import requests
import json

def chat_with_mistral(user_message, history):
    """Envía el mensaje a Ollama/Mistral y devuelve la respuesta."""
    
    messages = []
    
    # System prompt
    messages.append({
        "role": "system",
        "content": (
            "You are a helpful financial analyst assistant. "
            "Answer questions clearly and concisely. "
            "If you don't know something, say so."
        )
    })
    
    # Agregar historial previo (formato por defecto: lista de tuplas [user, assistant])
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})
    
    # Agregar mensaje actual del usuario
    messages.append({"role": "user", "content": user_message})
    
    # Llamar a Ollama API
    try:
        response = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": "mistral",
                "messages": messages,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        result = response.json()
        return result["message"]["content"]
    
    except requests.exceptions.ConnectionError:
        return "❌ Error: No se pudo conectar a Ollama. Asegúrate de que esté corriendo (`ollama serve`)."
    except requests.exceptions.Timeout:
        return "⏳ Error: Ollama tardó demasiado en responder. Intenta de nuevo."
    except Exception as e:
        return f"❌ Error inesperado: {str(e)}"

# Crear la interfaz (sin type="messages")
demo = gr.ChatInterface(
    fn=chat_with_mistral,
    title="📊 SEC Filing Assistant",
    description="Chat con Mistral vía Ollama — Fase 2 del proyecto DAPT",
    examples=[
        "What is a 10-K filing?",
        "Explain the difference between 10-K and 10-Q reports.",
        "What are the main sections of a quarterly earnings report?",
        "Summarize what revenue recognition means in SEC filings."
    ]
)

demo.launch(share=False)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
